## 七种流式输出模式

这和langgraph的流式输出模式相同，因为Agent底层就是一个ReAct Agent graph。

| 模式 | 输出内容 | 使用场景 |
| ---- | ---- | ---- |
| values | 每个步骤执行后，都会输出完整的状态信息 | 适用于每一步都要获取完整状态、状态持久化场景 |
| updates（默认） | 每个步骤执行后，只增量更新状态中发生变化的内容 | 用于监控Agent执行进度，例如观察Agent决定调用工具、工具执行结果等步骤 |
| messages | 输出流式返回的Token以及相关的元数据（如：来自哪个节点model/tool） | 实现类似ChatGPT的打字机效果，为聊天机器人等交互式应用提供最佳的实时体验 |
| tasks | 输出当前task任务开始和结束的时间，包含任务的结果和错误信息 | 该模式用于监控任务的生命周期 |
| debug | 与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result） | 该模式用于调试、监控task任务的生命周期 |
| checkpoints | 当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态 | 用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景 |
| custom | 通过get_stream_writer在工具或节点内部自定义发送的数据 | 用于输出业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标 |

我们可以根据不同的目标来选择不同的输出模式。例如：
- 实现**实时对话交互**，优先选择messages模式；
- 观察Agent的**思考与执行步骤**，优先选择updates模式；
- 需要查看**每一步状态**优先选择values/tasks/debug模式；
- 在工具执行时**输出自定义业务**日志优先选择custom模式。

## values模式

当stream_mode设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}

@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }

# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)

for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="values"
):
    rprint(chunk)
    print(len(chunk["messages"]))
    print(type(chunk))
    print("-" * 50)

{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='d30b2f47-fd04-4519-a3a6-059f10b90b73'
        )
    ]
}

<class 'dict'>
--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='d30b2f47-fd04-4519-a3a6-059f10b90b73'
        ),
        AIMessage(
            content='好的，我来为您同时查询这三方面的信息。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '用户希望查询客户ID为 CUST123456 
的个人信息、历史订单和可用优惠。我需要同时调用三个工具来获取这些信息。这三个工具之间没有依赖关系，可以同时调用。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 152,
                    'prompt_tokens': 456,
                    'total_tokens': 608,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 41,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 72
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '96a8266c-2ab2-4450-a4d9-306909817b34',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7394-1d7a-7b11-93b0-e36c99d578db-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_00_Czjvhmpy2Iwxisu8RAOc7573',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_01_u2Sls4XYHYqQSKIIG2xw0482',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_3CcVJy3hEC1sQFM8XrZ95265',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 456,
                'output_tokens': 152,
                'total_tokens': 608,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {'reasoning': 41}
            }
        )
    ]
}

<class 'dict'>
--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='d30b2f47-fd04-4519-a3a6-059f10b90b73'
        ),
        AIMessage(
            content='好的，我来为您同时查询这三方面的信息。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '用户希望查询客户ID为 CUST123456 
的个人信息、历史订单和可用优惠。我需要同时调用三个工具来获取这些信息。这三个工具之间没有依赖关系，可以同时调用。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 152,
                    'prompt_tokens': 456,
                    'total_tokens': 608,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 41,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 72
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '96a8266c-2ab2-4450-a4d9-306909817b34',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7394-1d7a-7b11-93b0-e36c99d578db-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_00_Czjvhmpy2Iwxisu8RAOc7573',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_01_u2Sls4XYHYqQSKIIG2xw0482',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_3CcVJy3hEC1sQFM8XrZ95265',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 456,
                'output_tokens': 152,
                'total_tokens': 608,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {'reasoning': 41}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='f40377a3-6f7f-4663-a50b-9969e8585737',
            tool_call_id='call_00_Czjvhmpy2Iwxisu8RAOc7573'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='17816909-5aac-436a-a0bf-df0de6d09238',
            tool_call_id='call_01_u2Sls4XYHYqQSKIIG2xw0482'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='1d88d9ad-777a-48ce-8ea7-d82d4147e1ee',
            tool_call_id='call_02_3CcVJy3hEC1sQFM8XrZ95265'
        )
    ]
}

<class 'dict'>
--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='d30b2f47-fd04-4519-a3a6-059f10b90b73'
        ),
        AIMessage(
            content='好的，我来为您同时查询这三方面的信息。',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '用户希望查询客户ID为 CUST123456 
的个人信息、历史订单和可用优惠。我需要同时调用三个工具来获取这些信息。这三个工具之间没有依赖关系，可以同时调用。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 152,
                    'prompt_tokens': 456,
                    'total_tokens': 608,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 41,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                    'prompt_cache_hit_tokens': 384,
                    'prompt_cache_miss_tokens': 72
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '96a8266c-2ab2-4450-a4d9-306909817b34',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7394-1d7a-7b11-93b0-e36c99d578db-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_00_Czjvhmpy2Iwxisu8RAOc7573',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_01_u2Sls4XYHYqQSKIIG2xw0482',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_3CcVJy3hEC1sQFM8XrZ95265',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 456,
                'output_tokens': 152,
                'total_tokens': 608,
                'input_token_details': {'cache_read': 384},
                'output_token_details': {'reasoning': 41}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='f40377a3-6f7f-4663-a50b-9969e8585737',
            tool_call_id='call_00_Czjvhmpy2Iwxisu8RAOc7573'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='17816909-5aac-436a-a0bf-df0de6d09238',
            tool_call_id='call_01_u2Sls4XYHYqQSKIIG2xw0482'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='1d88d9ad-777a-48ce-8ea7-d82d4147e1ee',
            tool_call_id='call_02_3CcVJy3hEC1sQFM8XrZ95265'
        ),
        AIMessage(
            content='以下是客户 **CUST123456** 的完整信息汇总：\n\n---\n\n### 👤 个人信息\n| 项目 | 内容 
|\n|------|------|\n| **姓名** | 张三 |\n| **会员等级** | 🏆 **VIP** |\n| **加入日期** | 2023年1月15日 |\n\n### 📦 
历史订单\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 单 |\n| **累计消费金额** | **¥25,800.00** |\n\n### 
🎉 当前可用优惠活动\n| 优惠活动 | 有效截止日期 |\n|---------|------------|\n| ❶ **老用户优惠** | 2027年1月31日 |\n|
❷ **会员专属折扣** | 2027年1月31日 |\n\n---\n\n**总结：** 张三先生是我们平台的 **VIP 
会员**，自2023年1月加入以来已累计消费 **25,800元**（共15笔订单），目前有两项优惠活动可供使用，有效期均至 
**2027年1月

<class 'dict'>
--------------------------------------------------


## updates模式
这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [4]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content='好的，我来为您查询客户 CUST123456 的相关信息。',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'用户想查询客户ID为CUST123456的个人信息、历史订单和可用优惠。这三个查询之间没有依赖关系，可以并行调用。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 145,
                        'prompt_tokens': 456,
                        'total_tokens': 601,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 30,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
                        'prompt_cache_hit_tokens': 384,
                        'prompt_cache_miss_tokens': 72
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '62b1495b-c853-4de1-95db-2efcfe90a153',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--019f7396-fd14-79b0-b179-087c88d7aada-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_00_tTaJHBM49IYATwhkUqOu6189',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_01_Zvj3hANXzVEN0bO4Rg4E7642',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_nmGAUg7LBSelhyDRKqSQ3158',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 456,
                    'output_tokens': 145,
                    'total_tokens': 601,
                    'input_token_details': {'cache_read': 384},
                    'output_token_details': {'reasoning': 30}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='c38e768b-3528-4e44-af03-4e26288b8cfb',
                tool_call_id='call_01_Zvj3hANXzVEN0bO4Rg4E7642'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='776d96ab-fe27-4652-a832-cfab09d938fe',
                tool_call_id='call_00_tTaJHBM49IYATwhkUqOu6189'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='acf01a26-7712-452e-a063-21d920d08962',
                tool_call_id='call_02_nmGAUg7LBSelhyDRKqSQ3158'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='以下是客户 **CUST123456** 的完整信息汇总：\n\n---\n\n### 👤 个人信息\n| 项目 | 内容 
|\n|------|------|\n| **姓名** | 张三 |\n| **会员等级** | VIP |\n| **加入日期** | 2023-01-15 |\n\n---\n\n### 📦 
历史订单\n| 项目 | 数据 |\n|------|------|\n| **总订单数** | 15 笔 |\n| **累计消费金额** | ¥25,800.00 
|\n\n---\n\n### 🎉 当前可用优惠活动\n| 活动名称 | 有效期至 |\n|---------|---------|\n| 老用户优惠 | 2027-01-31 |\n|
会员专属折扣 | 2027-01-31 |\n\n---\n\n**小结：** 张三先生是 **VIP 会员**，自2023年1月加入以来共完成了 
**15笔订单**，累计消费 **¥25,800**。当前有 **"老用户优惠"** 和 **"会员专属折扣"** 两项促销活动可用，有效期至 
**2027年1月31日**。请问还有什么需要帮忙的吗？😊',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': '好的，三个查询都返回了结果。现在我来整理一下信息给用户。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 264,
                        'prompt_tokens': 698,
                        'total_tokens': 962,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 17,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                        'prompt_cache_hit_tokens': 512,
                        'prompt_cache_miss_tokens': 186
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                    'id': '4503d199-edb2-4eb9-8e55-50911e05e654',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019f7397-03a1-7913-9909-281996ff288c-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 698,
                    'output_tokens': 264,
                    'total_tokens': 962,
                    'input_token_details': {'cache_read': 512},
                    'output_token_details': {'reasoning': 17}
                }
            )
        ]
    }
}

--------------------------------------------------


## messages模式

出流式返回的Token以及相关的元数据（

In [6]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="messages"
):
    print(chunk)
    print("-" * 50)

(AIMessageChunk(content='', additional_kwargs={'reasoning_content': ''}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019f73f2-bfc3-7c73-9800-3009fa3824bb', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:36d59bde-7bf6-4227-c625-0d80b8816798', 'checkpoint_ns': 'model:36d59bde-7bf6-4227-c625-0d80b8816798', 'ls_provider': 'deepseek', 'ls_model_name': 'deepseek-v4-flash', 'ls_model_type': 'chat', 'ls_temperature': None, 'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-openai': '1.3.3'}})
--------------------------------------------------
(AIMessageChunk(content='', additional_kwargs={'reasoning_content': '用户'}, response_metadata={'model_provider': 'deepseek'}, id='lc_run--019f73f2-bfc3-7c73-9800-3009fa3824bb', tool_calls=

## tasks模式

输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生
命周期。 注意观察如何定义一个任务

In [8]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="tasks"
):
    print(chunk)
    print("-" * 50)

{'id': 'fe597ee2-e456-df58-59e0-c7578073f21a', 'name': 'model', 'input': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠', additional_kwargs={}, response_metadata={}, id='6d3bea08-f29f-4342-ae49-c01a5adf9084')]}, 'triggers': ('branch:to:model',), 'metadata': {'ls_integration': 'langchain_create_agent'}}
--------------------------------------------------
{'id': 'fe597ee2-e456-df58-59e0-c7578073f21a', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content='我来为您同时查询该客户的个人信息、历史订单和当前可用优惠活动。', additional_kwargs={'refusal': None, 'reasoning_content': '用户想要查询客户ID为CUST123456的个人信息、历史订单和可用优惠。这三个查询是独立的，可以同时进行。让我调用三个工具。'}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 456, 'total_tokens': 606, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 33, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hi

## debug模式

与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式
用于调试、监控task任务的生命周期。

In [9]:
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
    stream_mode="debug"
):
    print(chunk)
    print("-" * 50)

{'step': 1, 'timestamp': '2026-07-18T06:41:55.699744+00:00', 'type': 'task', 'payload': {'id': 'c8e24e1e-4b65-1ab9-15de-bec1169d6098', 'name': 'model', 'input': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠', additional_kwargs={}, response_metadata={}, id='0d585c06-0222-475e-8cfb-7430213c3fbc')]}, 'triggers': ('branch:to:model',), 'metadata': {'ls_integration': 'langchain_create_agent'}}}
--------------------------------------------------
{'step': 1, 'timestamp': '2026-07-18T06:41:57.430139+00:00', 'type': 'task_result', 'payload': {'id': 'c8e24e1e-4b65-1ab9-15de-bec1169d6098', 'name': 'model', 'error': None, 'result': {'messages': [AIMessage(content='好的，我同时为您查询这三项信息。', additional_kwargs={'refusal': None, 'reasoning_content': '用户需要查询客户ID为 CUST123456 的个人信息、历史订单和当前可用优惠。这需要调用三个工具，它们是独立的，可以并行调用。'}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 456, 'total_tokens': 601, 'completion_tokens_details': {'accepted_prediction_tokens': None,

## checkpoint模式

每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()


customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user", "content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
    config=config,
    stream_mode='checkpoints'
):
    checkpoint_count +=1
    print(f"检查点#{checkpoint_count}")
    print(chunk)
    print("_" * 50)


检查点#1
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f182748-15ab-6d7d-bfff-d7c036696bf6'}}, 'parent_config': None, 'values': {'messages': []}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '62faab83-376a-f66b-7889-456abb2963a0', 'name': '__start__', 'interrupts': (), 'state': None}]}
__________________________________________________
检查点#2
{'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f182748-15b1-66ef-8000-7e3da3d60e85'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'session01', 'checkpoint_id': '1f182748-15ab-6d7d-bfff-d7c036696bf6'}}, 'values': {'messages': [HumanMessage(content='查询客户ID为 CUST123456 的完整信息和可用优惠', additional_kwargs={}, response_metadata={}, id='c878cdce-6e6e-4b15-a026-863bcae8dd34')]}, 'metadata': {'source': 'loop', 'step': 0, 'parents': {}}, 'next': ['model'], 'tasks': [{'id': '91349d0e-11ce-7c

##  custom模式

开发者通过get_stream_writer在工具或节点内部自定义发送的数据，用于输出业务逻辑相关的送
度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [11]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告", "message": f"生成销售报告进度百分比: {i * 25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})

    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")

    return "当前库存量为10000件，库存充足，无异常"

# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
    {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
    stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比: 25%'}
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比: 50%'}
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比: 75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------
